# Week 1: Data Exploration - Diabetic Retinopathy Detection

**Dataset:** EyePACS + APTOS + Messidor Combined DR

**Source:** https://www.kaggle.com/datasets/ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')

## A. Load Data
Dataset uses ImageFolder structure: class labels are folder names (0-4).

In [ ]:
BASE_DIR = '../data/raw/dr_unified_v2/dr_unified_v2'
CLASS_NAMES = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative'}

records = []
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(BASE_DIR, split)
    for class_folder in sorted(os.listdir(split_dir)):
        class_path = os.path.join(split_dir, class_folder)
        if not os.path.isdir(class_path): continue
        severity = int(class_folder)
        for img_file in os.listdir(class_path):
            if img_file.lower().endswith(('.jpeg', '.jpg', '.png')):
                records.append({'image_id': img_file, 'severity': severity, 'split': split, 'image_path': os.path.join(class_path, img_file)})

df = pd.DataFrame(records)
print(f'Total images: {len(df)}')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Splits: {dict(df["split"].value_counts())}')
print(f'\nMissing values:\n{df.isnull().sum()}')
df.head(10)

## B. Class Distribution Analysis

In [ ]:
counts = df['severity'].value_counts().sort_index()
percentages = (counts / len(df)) * 100
imbalance_ratio = counts.max() / counts.min()

for idx, count in counts.items():
    name = CLASS_NAMES.get(idx, f'Class {idx}')
    print(f'Class {idx} ({name}): {count} images ({percentages[idx]:.2f}%)')
print(f'\nImbalance ratio: {imbalance_ratio:.2f}x')

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar([f'Class {i}\n({CLASS_NAMES.get(i,"")})' for i in counts.index], counts.values, color=sns.color_palette('viridis', len(counts)))
for bar, pct in zip(bars, percentages.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts.values)*0.01, f'{pct:.1f}%', ha='center', fontsize=11)
ax.set_title('Diabetic Retinopathy - Class Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Images')
plt.tight_layout()
plt.savefig('../reports/figures/01_class_distribution.png', dpi=150)
plt.show()

## C. Image Statistics (sample 500 images)

In [ ]:
np.random.seed(42)
sample_paths = np.random.choice(df['image_path'].tolist(), min(500, len(df)), replace=False)

heights, widths, channels_list = [], [], []
corrupted = 0
for p in sample_paths:
    img = cv2.imread(p)
    if img is None:
        corrupted += 1
        continue
    h, w, c = img.shape
    heights.append(h); widths.append(w); channels_list.append(c)

print(f'Sampled: {len(sample_paths)}')
print(f'Valid: {len(sample_paths)-corrupted} | Corrupted: {corrupted}')
print(f'Mean dims: {np.mean(heights):.0f} x {np.mean(widths):.0f}')
print(f'Min dims:  {np.min(heights)} x {np.min(widths)}')
print(f'Max dims:  {np.max(heights)} x {np.max(widths)}')
print(f'Data quality: {((len(sample_paths)-corrupted)/len(sample_paths))*100:.1f}%')

## D. Data Quality Checks & Visualizations

In [ ]:
# Sample images per class
fig, axes = plt.subplots(5, 5, figsize=(18, 18))
fig.suptitle('Sample Images per DR Severity Class', fontsize=16, fontweight='bold')
for i in range(5):
    class_rows = df[df['severity'] == i]
    samples = class_rows.sample(min(5, len(class_rows)), random_state=42)
    for j, (_, row) in enumerate(samples.iterrows()):
        if j >= 5: break
        img = cv2.imread(row['image_path'])
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (256, 256))
            axes[i, j].imshow(img)
        axes[i, j].set_title(f'Class {i} ({CLASS_NAMES.get(i,"")})', fontsize=9)
        axes[i, j].axis('off')
plt.tight_layout()
plt.savefig('../reports/figures/02_sample_images_per_class.png', dpi=150)
plt.show()

# Image dimensions
fig, axes_dim = plt.subplots(1, 2, figsize=(14, 5))
axes_dim[0].hist(heights, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes_dim[0].set_title('Height Distribution')
axes_dim[1].hist(widths, bins=30, color='coral', edgecolor='white', alpha=0.8)
axes_dim[1].set_title('Width Distribution')
fig.suptitle('Image Dimensions Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/03_image_dimensions.png', dpi=150)
plt.show()

# Pie chart
fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(counts.values, labels=[f'Class {i}\n{CLASS_NAMES.get(i,"")}' for i in counts.index], autopct='%1.1f%%', colors=sns.color_palette('viridis', len(counts)), startangle=90)
ax.set_title('Class Imbalance Analysis', fontsize=14, fontweight='bold')
plt.savefig('../reports/figures/04_class_imbalance.png', dpi=150)
plt.show()

# Data quality bar
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(['Valid', 'Corrupted'], [len(sample_paths)-corrupted, corrupted], color=['#2ecc71', '#e74c3c'])
ax.set_title('Data Quality Breakdown', fontsize=14, fontweight='bold')
plt.savefig('../reports/figures/05_data_quality.png', dpi=150)
plt.show()

## E. Create Stratified Splits

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['severity'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['severity'], random_state=42)

for name, split in [('train', train_df), ('val', val_df), ('test', test_df)]:
    split.to_csv(f'../data/labels/{name}.csv', index=False)
    print(f'{name}.csv: {len(split)} rows')
    print(f'  Distribution: {dict(split["severity"].value_counts().sort_index())}')